# 19 · Rayleigh–Bénard plumes — HDiv-HDG & HDG 🔥🌀

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⌛ ~1 min](/lite/notebooks/index.html?path=19-rayleigh-benard-hdg.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/19-rayleigh-benard-hdg.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time). Use the ⚙ **View options** to
switch story / quizzes / gimmicks on or off.</sub>
:::

:::{dropdown} 🎭 Story — the Beast breathes fire
:class: storytelling

*Deep in its body the Beast hoards fire-energy. Heat it from below and the warmth will
not sit still: it **rises in plumes**, cold sinks to take its place, and the whole channel
stirs itself into rolling cells. Today you light that fire — and tame it with the most
elegant tools in the kit.*
:::

**Buoyancy-driven convection**: a long closed channel heated from below (hot floor, cold
ceiling, insulated side walls). Above a critical temperature difference the still fluid
becomes unstable and **thermal plumes** form — the classic **Rayleigh–Bénard** problem.
We solve the **Boussinesq** equations with two of NGSolve's most powerful flow tools:

* an **H(div)-conforming HDG** velocity that is **exactly divergence-free**, and
* a scalar **HDG** temperature,

advanced in time by an **IMEX** scheme that keeps every implicit operator *constant* — so
we factorise **once** with the CI-safe `sparsecholesky` and reuse it every step.

In non-dimensional form (velocity scaled by the thermal diffusion speed, $T\in[0,1]$):
$$ \tfrac{1}{Pr}\bigl(\partial_t\mathbf u + (\mathbf u\!\cdot\!\nabla)\mathbf u\bigr)
   = \Delta\mathbf u - \nabla p + Ra\,T\,\mathbf e_y,\quad \nabla\!\cdot\!\mathbf u=0,\qquad
   \partial_t T + \mathbf u\!\cdot\!\nabla T = \Delta T . $$

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from netgen.occ import *
import numpy as np
from ngsolve.webgui import Draw

# A long, shallow box — room for several convection cells.
L, H = 4.0, 1.0
Ra, Pr = 1e4, 0.71                                   # Rayleigh & Prandtl numbers
order, maxh, dt = 2, 0.10, 5e-4

rect = MoveTo(0, 0).Rectangle(L, H).Face()
for e, nm in [(rect.edges.Min(Y), "bot"), (rect.edges.Max(Y), "top"),
              (rect.edges.Min(X), "left"), (rect.edges.Max(X), "right")]:
    e.name = nm
mesh = Mesh(OCCGeometry(rect, dim=2).GenerateMesh(maxh=maxh))
print(f"channel mesh: {mesh.ne} elements")

n = specialcf.normal(2)
h = specialcf.mesh_size
def tang(w): return w - (w * n) * n                  # tangential part on a facet
dS = dx(element_boundary=True)
alpha = 4                                            # HDG interior-penalty parameter

## 1. The velocity space — H(div)-conforming HDG

We discretise the velocity in **`HDiv`**: its **normal** component is continuous across
facets. Pair it with a **pressure** in `L2` of one order lower and the discrete velocity
is **pointwise divergence-free** — mass is conserved *exactly*, not just approximately.
H(div) elements have no built-in *tangential* continuity, so we add a small **facet
unknown** (`TangentialFacetFESpace`) and tie the tangential velocity together **weakly**,
HDG-style (consistency + symmetry + an interior penalty). No-slip walls fix both pieces.

In [ ]:
V    = HDiv(mesh, order=order, dirichlet="bot|top|left|right")          # normal-continuous
Vhat = TangentialFacetFESpace(mesh, order=order, dirichlet="bot|top|left|right")  # tangential trace
Q    = L2(mesh, order=order - 1)                                        # pressure
X = V * Vhat * Q
(u, uhat, p), (v, vhat, q) = X.TnT()
gfu = GridFunction(X); velocity = gfu.components[0]
print(f"velocity system: {X.ndof} dofs  (exactly divergence-free)")

## 2. The implicit Stokes operator — assembled once

IMEX treats the *stiff, linear* pieces **implicitly** and the *cheap, nonlinear* pieces
(convection, buoyancy) **explicitly**. For the momentum equation the implicit part is a
**generalised Stokes** operator — mass $\tfrac{1}{Pr\,\Delta t}$ + viscous HDG +
incompressibility — which does **not change** from step to step. A tiny $-\varepsilon\,pq$
regularises the zero pressure block so the **symmetric indefinite** system factorises with
`sparsecholesky` (the CI-safe direct solver, unit 7); we build that factorisation **once**.

In [ ]:
eps = 1e-9
a = BilinearForm(X, symmetric=True)
a += 1 / (Pr * dt) * InnerProduct(u, v) * dx
a += InnerProduct(Grad(u), Grad(v)) * dx
a += (-InnerProduct(Grad(u) * n, tang(v - vhat)) - InnerProduct(Grad(v) * n, tang(u - uhat))
      + alpha * order * order / h * InnerProduct(tang(u - uhat), tang(v - vhat))) * dS
a += (-div(u) * q - div(v) * p - eps * p * q) * dx
a.Assemble()
ainv = a.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")            # factor ONCE, reuse
mass_u = BilinearForm(1 / (Pr * dt) * InnerProduct(u, v) * dx, symmetric=True).Assemble()

## 3. The temperature — a scalar HDG

Temperature lives in **`L2`** with a **facet** unknown for the diffusion (HDG again):
implicit diffusion + mass, also constant in time, also factored once. The **hot floor**
($T=1$) and **cold ceiling** ($T=0$) are Dirichlet data on the facet trace; the side walls
are **insulated** (a natural, do-nothing Neumann condition). The starting field is the pure
conduction profile $1-y$ with a small wiggle to trigger the instability.

In [ ]:
W    = L2(mesh, order=order)
What = FacetFESpace(mesh, order=order, dirichlet="bot|top")
Y_ = W * What
(T, That), (s, shat) = Y_.TnT()
gfT = GridFunction(Y_); temp = gfT.components[0]

aT = BilinearForm(Y_, symmetric=True)
aT += 1 / dt * T * s * dx + Grad(T) * Grad(s) * dx
aT += (-Grad(T) * n * (s - shat) - Grad(s) * n * (T - That)
       + alpha * order * order / h * (T - That) * (s - shat)) * dS
aT.Assemble()
aTinv = aT.mat.Inverse(Y_.FreeDofs(), inverse="sparsecholesky")
mass_T = BilinearForm(1 / dt * T * s * dx, symmetric=True).Assemble()

gfT.components[0].Set(1 - y + 0.1 * sin(4 * pi * x / L) * sin(pi * y))   # conduction + seed
gfT.components[1].Set(1 - y, definedon=mesh.Boundaries("bot|top"))       # hot floor / cold ceiling

## 4. The explicit pieces — upwind transport & buoyancy

Convection $\mathbf u\!\cdot\!\nabla T$ is handled **explicitly** with an **upwind DG** flux
(cheap, no matrix to factor — we only ever apply it). Buoyancy $Ra\,T\,\mathbf e_y$ enters
the momentum right-hand side. (At this Rayleigh number the momentum self-advection is small;
the plume dynamics live in the temperature transport + buoyancy coupling.)

In [ ]:
un = velocity * n
convT = BilinearForm(Y_, nonassemble=True)                              # only .Apply() is used
convT += -T * (velocity * Grad(s)) * dx
convT += un * IfPos(un, T, T.Other()) * (s - s.Other()) * dx(skeleton=True)
buoyancy = LinearForm(Ra * temp * v[1] * dx)                            # uses the current temp
resT = gfT.vec.CreateVector()

## 5. Step in time — and watch the plumes grow

One IMEX step is two reused back-substitutions: **(1)** advance the temperature
(explicit upwind convection on the right, implicit diffusion via `aTinv`), keeping the
Dirichlet data with a residual update; **(2)** advance the velocity (explicit buoyancy from
the fresh temperature, implicit Stokes via `ainv`). We snapshot the temperature for an
animation and track the **Nusselt number** $Nu = 1 + \langle u_y T\rangle$ — the convective
boost to the vertical heat transport (it climbs from 1 as the cells switch on).

In [ ]:
tend = 0.5
nsteps = int(tend / dt + 0.5)
anim = GridFunction(W, multidim=0)                                      # temperature movie
nus, ts = [], []
with TaskManager():
    for step in range(1, nsteps + 1):
        convT.Apply(gfT.vec, resT)                                      # explicit transport
        gfT.vec.data += aTinv * ((mass_T.mat * gfT.vec - resT).Evaluate() - aT.mat * gfT.vec)
        buoyancy.Assemble()                                            # explicit buoyancy
        gfu.vec.data = ainv * (mass_u.mat * gfu.vec + buoyancy.vec)     # implicit Stokes
        if step % 25 == 0:
            anim.AddMultiDimComponent(temp.vec)
            ts.append(step * dt)
            nus.append(1 + Integrate(velocity[1] * temp, mesh) / (L * H))
print(f"done: {nsteps} steps,  ‖div u‖ = {sqrt(Integrate(div(velocity)**2, mesh)):.1e}  "
      f"(exactly div-free),  final Nu ≈ {nus[-1]:.2f}")

## 6. The result — rising and sinking plumes

Drag the **multidim** slider (or press play) to watch the conduction layer break into
**plumes**: hot fingers rise off the floor, cold fingers drop from the ceiling, and they
organise into counter-rotating **convection cells** along the channel.

In [ ]:
Draw(anim, mesh, "temperature", interpolate_multidim=True, animate=True,
     min=0, max=1, autoscale=False, deformation=False)

In [ ]:
Draw(velocity, mesh, "velocity", vectors={"grid_size": 40})

The Nusselt number rises from $1$ (pure conduction) as the plumes switch on the convective
heat transport:

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(ts, nus); plt.axhline(1.0, ls="--", c="gray")
plt.xlabel("time"); plt.ylabel("Nusselt number  $Nu$")
plt.title(f"convective heat transport switches on (Ra = {Ra:.0e})")
plt.grid(alpha=0.3); plt.tight_layout()

:::{dropdown} 📚 Further reading
:class: further-reading

- **H(div)-conforming HDG for incompressible flow** — exactly divergence-free, pressure-robust:
  the method behind this notebook (Lehrenfeld & Schöberl). See the ngs24 *Normal-continuous /
  H(div)-conforming HDG* tutorials: [docu.ngsolve.org/ngs24](https://docu.ngsolve.org/ngs24/intro.html).
- **Incompressible Navier–Stokes in NGSolve** — i-tutorial
  [3.2](https://docu.ngsolve.org/latest/i-tutorials/unit-3.2-navierstokes/navierstokes.html).
- **Benchmarks** — the steady [de Vahl Davis](https://doi.org/10.1002/fld.1650030305) differentially
  heated cavity (Pr = 0.71), and Rayleigh–Bénard convection in rectangular cavities.
:::

:::{dropdown} 🧠 Quiz — why factor only once?
:class: quiz
Because **IMEX** puts everything that changes from step to step (the nonlinear convection,
the buoyancy load) on the **right-hand side**, while the implicit operators — generalised
Stokes and the temperature diffusion — depend only on `dt`, the mesh and the parameters,
which are **fixed**. A constant matrix means **one** `sparsecholesky` factorisation, reused
as a cheap back-substitution every step — fast *and* CI-safe, no indefinite re-solve.
:::

**That is the toolkit:** an exactly divergence-free H(div)-HDG velocity, a scalar HDG
temperature, coupled through buoyancy and marched with a factor-once IMEX scheme — enough
to set a whole channel convecting. The expedition continues. ☕

In [ ]:
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _nb, _title = "18-outlook-unfitted", "18 · Outlook — unfitted FEM with ngsxfem 🫧"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next:** [" + _title + "](" + _u + ")"))